In [2]:
import subprocess
subprocess.run(["pip", "install", "einops", "nibabel", "-q"])

import os, json, time, gc, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from scipy.ndimage import zoom, rotate, gaussian_filter, map_coordinates
from scipy.ndimage import label as scipy_label
import nibabel as nib
from einops import rearrange
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla P100-PCIE-16GB
VRAM: 17.1 GB


In [3]:
DATA_DIR = None
for root in ["/kaggle/input"]:
    if not os.path.exists(root): continue
    for d in os.listdir(root):
        full = os.path.join(root, d)
        for dp, dn, fn in os.walk(full):
            if "dataset.json" in fn and os.path.exists(os.path.join(dp, "imagesTr")):
                DATA_DIR = dp
                print(f"✅ DATA_DIR = {DATA_DIR}")
                break
        if DATA_DIR: break

if not DATA_DIR:
    print("❌ Not found! Set manually")

✅ DATA_DIR = /kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas


In [4]:
CONFIG = {
    "data_dir": DATA_DIR,
    "cache_dir": "/kaggle/working/cache_s2_roi",
    "output_dir": "/kaggle/working",
    "patch_size": (64, 64, 64),
    "batch_size": 4,
    "accum_steps": 2,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "epochs": 250,
    "val_split": 0.15,
    "num_workers": 2,
    "resume_from": None,
    "target_spacing": (1.5, 1.5, 2.5),
    "hu_window": (-125, 275),
    "roi_margin": 15,
    "fg_sample_rate": 0.8,    # Higher: focus on tumor
}
os.makedirs(CONFIG["cache_dir"], exist_ok=True)
print("✅ Stage 2 Config ready")

✅ Stage 2 Config ready


In [2]:
import os
import shutil

def clear_working_dir(path='.'):
    for item in os.listdir(path):
        item_path = os.path.join(path, item)
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.unlink(item_path) # Delete file or link
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path) # Delete folder
        except Exception as e:
            print(f'Failed to delete {item_path}. Reason: {e}')
    
    print("✅ Working directory cleared.")

# Usage
clear_working_dir()

✅ Working directory cleared.


In [3]:
import os
import json
import numpy as np
import nibabel as nib
from scipy.ndimage import zoom

def get_bbox(mask, margin=15):
    coords = np.argwhere(mask > 0)
    if len(coords) == 0: return None
    mins = np.maximum(coords.min(0) - margin, 0)
    maxs = np.minimum(coords.max(0) + 1 + margin, mask.shape)
    return tuple(slice(mn, mx) for mn, mx in zip(mins, maxs))

def preprocess_rois_gt(data_dir, cache_dir, target_spacing, hu_window, margin):
    """Preprocess and crop ROIs using GROUND TRUTH bounding boxes"""
    json_path = os.path.join(data_dir, "dataset.json")
    with open(json_path) as f:
        meta = json.load(f)

    # Ensure cache directory exists
    os.makedirs(cache_dir, exist_ok=True)

    cases = []
    for case in meta.get("training", []):
        # Fix: Better path handling for './' prefixes
        img_rel = os.path.normpath(case["image"].replace("./", ""))
        lbl_rel = os.path.normpath(case["label"].replace("./", ""))
        
        img_path = os.path.join(data_dir, img_rel)
        lbl_path = os.path.join(data_dir, lbl_rel)

        # Fix: Check for .nii if .nii.gz doesn't exist (common in Kaggle datasets)
        if not os.path.exists(img_path):
            img_path = img_path.replace(".nii.gz", ".nii")
        if not os.path.exists(lbl_path):
            lbl_path = lbl_path.replace(".nii.gz", ".nii")

        if os.path.exists(img_path) and os.path.exists(lbl_path):
            cases.append((img_path, lbl_path))

    # Diagnostic print to ensure it's working now
    print(f"Preprocessing {len(cases)} ROIs using GT bounding boxes...")
    if len(cases) == 0:
        print("❌ Error: Still found 0 cases. Check if CONFIG['data_dir'] is correct.")
        return []

    roi_files = []
    for i, (img_path, lbl_path) in enumerate(cases):
        # Fix: Get case name regardless of .nii or .nii.gz extension
        case_name = os.path.basename(img_path).split('.')[0]
        cache_path = os.path.join(cache_dir, f"{case_name}_roi.npz")

        if os.path.exists(cache_path):
            roi_files.append(cache_path)
            continue

        # Load NIfTI files
        img_nifti = nib.load(img_path)
        ct = img_nifti.get_fdata().astype(np.float32)
        label = nib.load(lbl_path).get_fdata().astype(np.int32)
        spacing = np.array(img_nifti.header.get_zooms()[:3])

        # Resampling to target spacing
        scale = spacing / np.array(target_spacing)
        ct = zoom(ct, scale, order=1)
        label = zoom(label, scale, order=0)
        
        # HU Windowing
        ct = np.clip(ct, hu_window[0], hu_window[1])

        # Normalization (Foreground-based if possible)
        fg = label > 0
        if fg.sum() > 100:
            ct = (ct - ct[fg].mean()) / (ct[fg].std() + 1e-8)
        else:
            ct = (ct - ct.mean()) / (ct.std() + 1e-8)

        # Crop using GT bounding box
        bbox = get_bbox(label, margin)
        if bbox is None:
            # Fallback if label is empty (e.g., first 64 slices)
            ct_roi = ct[:64, :64, :64]
            label_roi = label[:64, :64, :64]
        else:
            ct_roi = ct[bbox]
            label_roi = label[bbox]

        # Save as compressed NumPy array
        np.savez_compressed(cache_path,
            ct_roi=ct_roi.astype(np.float32),
            label_roi=label_roi.astype(np.int8)
        )
        roi_files.append(cache_path)

        if (i+1) % 20 == 0:
            print(f"  [{i+1}/{len(cases)}] {case_name} ROI shape={ct_roi.shape}")

    print(f"✅ Done! {len(roi_files)} ROIs cached")
    return roi_files

# Usage
roi_files = preprocess_rois_gt(
    CONFIG["data_dir"], CONFIG["cache_dir"],
    CONFIG["target_spacing"], CONFIG["hu_window"], CONFIG["roi_margin"]
)

Preprocessing 281 ROIs using GT bounding boxes...
  [20/281] pancreas_328 ROI shape=(94, 69, 64)
  [40/281] pancreas_326 ROI shape=(116, 83, 58)
  [60/281] pancreas_292 ROI shape=(114, 69, 61)
  [80/281] pancreas_241 ROI shape=(103, 71, 65)
  [100/281] pancreas_296 ROI shape=(122, 92, 70)
  [120/281] pancreas_089 ROI shape=(105, 76, 61)
  [140/281] pancreas_229 ROI shape=(107, 68, 58)
  [160/281] pancreas_262 ROI shape=(109, 82, 62)
  [180/281] pancreas_028 ROI shape=(119, 78, 66)
  [200/281] pancreas_348 ROI shape=(97, 69, 60)
  [220/281] pancreas_004 ROI shape=(134, 92, 65)
  [240/281] pancreas_411 ROI shape=(106, 78, 65)
  [260/281] pancreas_088 ROI shape=(131, 84, 72)
  [280/281] pancreas_165 ROI shape=(100, 84, 62)
✅ Done! 281 ROIs cached


In [4]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Dropout3d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
        self.residual = nn.Identity() if in_ch == out_ch else nn.Conv3d(in_ch, out_ch, 1, bias=False)
    def forward(self, x):
        return self.conv(x) + self.residual(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.down = nn.Conv3d(in_ch, out_ch, 2, stride=2, bias=False)
        self.conv = ConvBlock(out_ch, out_ch, dropout)
    def forward(self, x):
        return self.conv(self.down(x))

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch, dropout)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class PatchEmbedding3D(nn.Module):
    def __init__(self, in_ch, embed_dim, patch_size=2):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = self.proj(x)
        B, C, D, H, W = x.shape
        x = rearrange(x, 'b c d h w -> b (d h w) c')
        return self.norm(x), (D, H, W)

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=6, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim), nn.Dropout(dropout),
        )
    def forward(self, x):
        h = self.norm1(x)
        x = x + self.attn(h, h, h)[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ViTBottleneck(nn.Module):
    def __init__(self, in_ch, embed_dim=384, heads=6, depth=3, patch_size=2, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding3D(in_ch, embed_dim, patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, 27, embed_dim) * 0.02)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, heads, dropout=dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.proj_back = nn.Linear(embed_dim, in_ch)
    def forward(self, x):
        B,C,D,H,W = x.shape
        tokens, (Dp,Hp,Wp) = self.patch_embed(x)
        N = tokens.shape[1]
        if N != 27:
            pos = F.interpolate(self.pos_embed.transpose(1,2), size=N, mode='linear', align_corners=False).transpose(1,2)
        else: pos = self.pos_embed
        tokens = self.blocks(tokens + pos)
        tokens = self.proj_back(self.norm(tokens))
        return rearrange(tokens, 'b (d h w) c -> b c d h w', d=Dp, h=Hp, w=Wp)

class ViTUNet(nn.Module):
    def __init__(self, in_ch=1, num_classes=2, base=24, vit_dim=384, vit_depth=3, vit_heads=6, deep_sup=True):
        super().__init__()
        self.deep_sup = deep_sup
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = DownBlock(base, base*2)
        self.enc3 = DownBlock(base*2, base*4, dropout=0.1)
        self.enc4 = DownBlock(base*4, base*8, dropout=0.1)
        self.down_bot = nn.Conv3d(base*8, base*8, 2, stride=2, bias=False)
        self.vit = ViTBottleneck(base*8, vit_dim, vit_heads, vit_depth, 2, 0.1)
        self.dec4 = UpBlock(base*8, base*8, base*8, dropout=0.1)
        self.dec3 = UpBlock(base*8, base*4, base*4, dropout=0.1)
        self.dec2 = UpBlock(base*4, base*2, base*2)
        self.dec1 = UpBlock(base*2, base, base)
        self.final = nn.Conv3d(base, num_classes, 1)
        if deep_sup:
            self.ds3 = nn.Conv3d(base*4, num_classes, 1)
            self.ds2 = nn.Conv3d(base*2, num_classes, 1)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
    def forward(self, x):
        s1 = self.enc1(x); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)
        b = self.vit(self.down_bot(s4))
        d4 = self.dec4(b, s4); d3 = self.dec3(d4, s3); d2 = self.dec2(d3, s2); d1 = self.dec1(d2, s1)
        out = self.final(d1)
        if self.deep_sup and self.training: return out, self.ds3(d3), self.ds2(d2)
        return out

class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__(); self.smooth = smooth
    def forward(self, pred, target):
        pred = F.softmax(pred, dim=1); nc = pred.shape[1]
        toh = F.one_hot(target.long(), nc).permute(0,4,1,2,3).float()
        scores = []
        for c in range(1, nc):
            p, t = pred[:,c].flatten(1), toh[:,c].flatten(1)
            scores.append((2*(p*t).sum(1)+self.smooth) / (p.sum(1)+t.sum(1)+self.smooth))
        return 1 - torch.stack(scores).mean()

class DiceCELoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__(); self.dice = SoftDiceLoss()
        w = torch.tensor(class_weights).float().cuda() if class_weights else None
        self.ce = nn.CrossEntropyLoss(weight=w)
    def forward(self, pred, target):
        return 0.5*self.dice(pred, target) + 0.5*self.ce(pred, target.long())

class DeepSupLoss(nn.Module):
    def __init__(self, base_loss, weights=[1.0, 0.5, 0.25]):
        super().__init__(); self.base = base_loss; self.weights = weights
    def forward(self, outputs, target):
        if not isinstance(outputs, tuple): return self.base(outputs, target)
        total = 0.0
        for pred, w in zip(outputs, self.weights):
            if pred.shape[2:] != target.shape[1:]:
                t = F.interpolate(target.unsqueeze(1).float(), size=pred.shape[2:], mode='nearest').squeeze(1).long()
            else: t = target
            total += w * self.base(pred, t)
        return total

def compute_dice(pred, target, num_classes=2):
    if pred.ndim == 5: pred = pred.argmax(dim=1)
    scores = {}
    for c in range(1, num_classes):
        p = (pred==c).float().flatten(1); t = (target==c).float().flatten(1)
        inter = (p*t).sum(1); union = p.sum(1) + t.sum(1); mask = union > 0
        if mask.sum() > 0: scores[f"c{c}"] = (2*inter[mask]/(union[mask]+1e-8)).mean().item()
        else: scores[f"c{c}"] = float('nan')
    valid = [v for v in scores.values() if v==v]
    scores["mean"] = sum(valid)/len(valid) if valid else 0.0
    return scores

class PolyLRScheduler:
    def __init__(self, opt, max_ep, power=0.9):
        self.opt=opt; self.max_ep=max_ep; self.power=power
        self.base_lrs=[pg['lr'] for pg in opt.param_groups]
    def step(self, ep):
        f = (1-ep/self.max_ep)**self.power
        for pg, blr in zip(self.opt.param_groups, self.base_lrs): pg['lr']=blr*f
    def state_dict(self): return {'base_lrs':self.base_lrs}
    def load_state_dict(self, s): self.base_lrs=s['base_lrs']

def keep_largest_component(mask):
    if mask.sum()==0: return mask
    labeled,num = scipy_label(mask)
    if num<=1: return mask
    sizes = [0]+[(labeled==i).sum() for i in range(1,num+1)]
    return (labeled==np.argmax(sizes)).astype(mask.dtype)

def elastic_deform(image, label, alpha=80, sigma=8):
    shape = image.shape
    dx = gaussian_filter(np.random.randn(*shape)*alpha, sigma, mode='constant')
    dy = gaussian_filter(np.random.randn(*shape)*alpha, sigma, mode='constant')
    dz = gaussian_filter(np.random.randn(*shape)*alpha, sigma, mode='constant')
    z,y,x = np.meshgrid(np.arange(shape[0]),np.arange(shape[1]),np.arange(shape[2]),indexing='ij')
    coords = [np.clip(z+dz,0,shape[0]-1),np.clip(y+dy,0,shape[1]-1),np.clip(x+dx,0,shape[2]-1)]
    return map_coordinates(image,coords,order=1,mode='reflect').astype(np.float32), \
           map_coordinates(label.astype(float),coords,order=0,mode='reflect').astype(np.int32)

def gamma_aug(image, gamma_range=(0.7,1.5)):
    mn=image.min(); s=image-mn+1e-8; mx=s.max()+1e-8
    g=np.random.uniform(*gamma_range); out=np.power(s/mx,g)*mx+mn
    if np.random.random()<0.5: out=-out+image.mean()*2
    return out.astype(np.float32)

print("✅ Model + Losses ready")

✅ Model + Losses ready


In [5]:
class ROIDataset(Dataset):
    """Stage 2 dataset from GT-cropped ROIs"""
    def __init__(self, files, patch_size=(64,64,64), augment=True, fg_rate=0.8):
        self.files = files; self.ps = patch_size
        self.augment = augment; self.fg_rate = fg_rate
        print(f"ROI Dataset: {len(files)} ROIs, patch={patch_size}")

    def __len__(self):
        return len(self.files) * 6

    def __getitem__(self, idx):
        data = np.load(self.files[idx % len(self.files)])
        ct = data['ct_roi']
        label = data['label_roi'].astype(np.int32)  # 0=bg, 1=pancreas, 2=tumor

        img, lbl = self._patch(ct, label)
        if self.augment: img, lbl = self._aug(img, lbl)
        return torch.from_numpy(img).unsqueeze(0).float(), torch.from_numpy(lbl).long()

    def _patch(self, image, label):
        D,H,W = image.shape; pd,ph,pw = self.ps
        if np.random.random() < self.fg_rate:
            fg = np.argwhere(label == 2)  # Tumor first
            if len(fg) < 5: fg = np.argwhere(label > 0)  # Then pancreas
            if len(fg) > 0:
                c = fg[np.random.randint(len(fg))] + np.random.randint(-pd//4, pd//4+1, size=3)
                d = np.clip(c[0]-pd//2, 0, max(0,D-pd))
                h = np.clip(c[1]-ph//2, 0, max(0,H-ph))
                w = np.clip(c[2]-pw//2, 0, max(0,W-pw))
            else:
                d,h,w = [np.random.randint(0,max(1,s-p+1)) for s,p in zip(image.shape,self.ps)]
        else:
            d,h,w = [np.random.randint(0,max(1,s-p+1)) for s,p in zip(image.shape,self.ps)]

        ip = image[d:d+pd, h:h+ph, w:w+pw]
        lp = label[d:d+pd, h:h+ph, w:w+pw]
        if ip.shape != tuple(self.ps):
            pi,pl = np.zeros(self.ps,np.float32), np.zeros(self.ps,np.int32)
            s=ip.shape; pi[:s[0],:s[1],:s[2]]=ip; pl[:s[0],:s[1],:s[2]]=lp
            ip,lp = pi,pl
        return ip, lp

    def _aug(self, img, lbl):
        for ax in range(3):
            if np.random.random()<0.5: img=np.flip(img,ax).copy(); lbl=np.flip(lbl,ax).copy()
        if np.random.random()<0.3:
            ang=np.random.uniform(-15,15); axes=[(0,1),(0,2),(1,2)][np.random.randint(3)]
            img=rotate(img,ang,axes=axes,reshape=False,order=1,mode='reflect')
            lbl=rotate(lbl.astype(float),ang,axes=axes,reshape=False,order=0,mode='reflect').astype(np.int32)
        if np.random.random()<0.2: img,lbl=elastic_deform(img,lbl)
        if np.random.random()<0.3: img=gamma_aug(img)
        if np.random.random()<0.2: img=img+np.random.normal(0,0.02,img.shape).astype(np.float32)
        if np.random.random()<0.3: img=img*np.random.uniform(0.9,1.1)+np.random.uniform(-0.1,0.1)
        if np.random.random()<0.15: img=gaussian_filter(img,sigma=np.random.uniform(0.5,1.0))
        return img.astype(np.float32), lbl.astype(np.int32)

# Test
td = ROIDataset(roi_files[:2], CONFIG["patch_size"])
i,l = td[0]
print(f"✅ Test: img={i.shape}, lbl={l.shape}, unique={l.unique().tolist()}")
print(f"   (0=bg, 1=pancreas, 2=tumor)")
del td

ROI Dataset: 2 ROIs, patch=(64, 64, 64)
✅ Test: img=torch.Size([1, 64, 64, 64]), lbl=torch.Size([64, 64, 64]), unique=[0, 1, 2]
   (0=bg, 1=pancreas, 2=tumor)


In [6]:
def train_stage2():
    np.random.seed(42)
    idx = np.random.permutation(len(roi_files))
    vs = int(len(roi_files) * CONFIG["val_split"])
    vf = [roi_files[i] for i in idx[:vs]]
    tf = [roi_files[i] for i in idx[vs:]]

    tds = ROIDataset(tf, CONFIG["patch_size"], True, CONFIG["fg_sample_rate"])
    vds = ROIDataset(vf, CONFIG["patch_size"], False, 0.5)
    tl = DataLoader(tds, CONFIG["batch_size"], True, num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True)
    vl = DataLoader(vds, CONFIG["batch_size"], False, num_workers=CONFIG["num_workers"], pin_memory=True)

    # 3 classes: bg, pancreas, tumor
    model = ViTUNet(1, 3, 24, 384, 3, 6).to(device)
    params = sum(p.numel() for p in model.parameters())/1e6
    print(f"Stage 2 Model: {params:.1f}M params (3 classes)")

    criterion = DeepSupLoss(DiceCELoss([0.15, 0.25, 0.60]))  # Heavy tumor weight
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = PolyLRScheduler(optimizer, CONFIG["epochs"])
    scaler = GradScaler()

    start_epoch, best_dice, best_tumor = 0, 0.0, 0.0
    hist = {"loss":[], "dice":[], "panc":[], "tumor":[]}

    if CONFIG["resume_from"] and os.path.exists(CONFIG["resume_from"]):
        ck = torch.load(CONFIG["resume_from"], map_location=device)
        model.load_state_dict(ck['model_state_dict'])
        optimizer.load_state_dict(ck['optimizer_state_dict'])
        scaler.load_state_dict(ck['scaler_state_dict'])
        start_epoch = ck['epoch']+1
        best_dice = ck['best_dice']; best_tumor = ck.get('best_tumor', 0)
        hist = ck.get('history', hist)
        print(f"✅ Resumed from epoch {start_epoch}")

    print(f"\n{'='*60}\n  STAGE 2: TUMOR SEGMENTATION (GT ROI)\n  {start_epoch} → {CONFIG['epochs']} epochs\n{'='*60}\n")

    for epoch in range(start_epoch, CONFIG["epochs"]):
        t0 = time.time(); model.train(); rl = 0; optimizer.zero_grad()

        for i, (imgs, lbls) in enumerate(tl):
            imgs, lbls = imgs.to(device), lbls.to(device)
            with autocast():
                loss = criterion(model(imgs), lbls) / CONFIG["accum_steps"]
            scaler.scale(loss).backward()
            if (i+1) % CONFIG["accum_steps"] == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            rl += loss.item() * CONFIG["accum_steps"]

        scheduler.step(epoch)
        tl_loss = rl / max(len(tl), 1)

        if epoch % 5 == 0 or epoch >= CONFIG["epochs"] - 50:
            model.eval()
            vd,vp,vt,cnt = 0,0,0,0
            with torch.no_grad():
                for imgs, lbls in vl:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    with autocast():
                        pred = model(imgs)
                        if isinstance(pred,tuple): pred=pred[0]
                    d = compute_dice(pred, lbls, 3)
                    vd += d["mean"]
                    vp += d.get("c1",0) if d.get("c1",0)==d.get("c1",0) else 0
                    vt += d.get("c2",0) if d.get("c2",0)==d.get("c2",0) else 0
                    cnt += 1
            val_dice = vd/max(cnt,1); val_panc = vp/max(cnt,1); val_tumor = vt/max(cnt,1)
        else:
            val_dice = hist["dice"][-1] if hist["dice"] else 0
            val_panc = hist["panc"][-1] if hist["panc"] else 0
            val_tumor = hist["tumor"][-1] if hist["tumor"] else 0

        hist["loss"].append(tl_loss); hist["dice"].append(val_dice)
        hist["panc"].append(val_panc); hist["tumor"].append(val_tumor)
        el = time.time()-t0; lr = optimizer.param_groups[0]['lr']

        star = ""
        if val_tumor > best_tumor:
            best_tumor = val_tumor; best_dice = val_dice
            torch.save({'epoch':epoch, 'model_state_dict':model.state_dict(),
                'optimizer_state_dict':optimizer.state_dict(), 'scaler_state_dict':scaler.state_dict(),
                'best_dice':best_dice, 'best_tumor':best_tumor, 'history':hist},
                os.path.join(CONFIG["output_dir"], "stage2_best.pth"))
            star = " ★"

        if epoch % 5 == 0 or star:
            print(f"E{epoch:03d} ({el:.0f}s) Loss:{tl_loss:.4f} Mean:{val_dice:.4f} "
                  f"Panc:{val_panc:.4f} Tumor:{val_tumor:.4f} LR:{lr:.6f}{star}")

        if (epoch+1) % 50 == 0:
            torch.save({'epoch':epoch, 'model_state_dict':model.state_dict(),
                'optimizer_state_dict':optimizer.state_dict(), 'scaler_state_dict':scaler.state_dict(),
                'best_dice':best_dice, 'best_tumor':best_tumor, 'history':hist},
                os.path.join(CONFIG["output_dir"], f"stage2_ep{epoch}.pth"))
            print(f"  💾 stage2_ep{epoch}.pth")

    print(f"\n✅ Done! Best Tumor: {best_tumor:.4f} | Best Mean: {best_dice:.4f}")
    return hist

history_s2 = train_stage2()

ROI Dataset: 239 ROIs, patch=(64, 64, 64)
ROI Dataset: 42 ROIs, patch=(64, 64, 64)
Stage 2 Model: 13.7M params (3 classes)

  STAGE 2: TUMOR SEGMENTATION (GT ROI)
  0 → 250 epochs



/tmp/ipykernel_55/1509307262.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_55/1509307262.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_55/1509307262.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


E000 (125s) Loss:1.1244 Mean:0.3086 Panc:0.4281 Tumor:0.1892 LR:0.001000 ★
E005 (122s) Loss:0.6469 Mean:0.4809 Panc:0.6607 Tumor:0.3010 LR:0.000982 ★
E010 (122s) Loss:0.5375 Mean:0.5034 Panc:0.7138 Tumor:0.2931 LR:0.000964
E015 (122s) Loss:0.4616 Mean:0.5308 Panc:0.7175 Tumor:0.3442 LR:0.000946 ★
E020 (122s) Loss:0.4153 Mean:0.5119 Panc:0.7333 Tumor:0.2906 LR:0.000928
E025 (122s) Loss:0.3833 Mean:0.5415 Panc:0.7405 Tumor:0.3425 LR:0.000910
E030 (122s) Loss:0.3487 Mean:0.5478 Panc:0.7446 Tumor:0.3510 LR:0.000891 ★
E035 (122s) Loss:0.3237 Mean:0.5503 Panc:0.7674 Tumor:0.3332 LR:0.000873
E040 (122s) Loss:0.3142 Mean:0.5887 Panc:0.7676 Tumor:0.4098 LR:0.000855 ★
E045 (122s) Loss:0.2944 Mean:0.5764 Panc:0.7600 Tumor:0.3928 LR:0.000836
  💾 stage2_ep49.pth
E050 (122s) Loss:0.2897 Mean:0.5648 Panc:0.7534 Tumor:0.3761 LR:0.000818
E055 (122s) Loss:0.2659 Mean:0.5861 Panc:0.7697 Tumor:0.4026 LR:0.000800
E060 (122s) Loss:0.2644 Mean:0.5755 Panc:0.7761 Tumor:0.3749 LR:0.000781
E065 (122s) Loss:0.24

In [ ]:
import os
from IPython.display import FileLink

# define the paths
best_path = "stage1_best.pth"       # The one with highest score
latest_path = "stage1_latest.pth"   # The most recent epoch (safety backup)

print(f"🔍 Looking for the best model...")

if os.path.exists(best_path):
    size_mb = os.path.getsize(best_path) / (1024 * 1024)
    print(f"✅ FOUND BEST MODEL: {best_path} ({size_mb:.2f} MB)")
    print("Click the blue link below to download it:")
    display(FileLink(best_path))
else:
    print(f"⚠️ 'stage1_best.pth' not found (maybe validation hasn't improved yet).")
    
    if os.path.exists(latest_path):
        print(f"⚠️ Downloading '{latest_path}' instead so you don't lose progress.")
        display(FileLink(latest_path))
    else:
        print("❌ CRITICAL: No model files found in /kaggle/working. Check your directory.")

In [1]:
from IPython.display import FileLink

# This creates a clickable link to your file
print("⬇️ Click the link below to download:")
FileLink(r'stage2_best.pth')

⬇️ Click the link below to download:


/kaggle/working/stage2_best.pth